<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula03b%20-%20gridsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [178]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy: {accuracy_score(y_test, y_pred)}")

accuracy: 0.9444444444444444


In [183]:
X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.2)

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_accuracy = 0
best_k = 0
~
# Validação interna
for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  X_train2_scaled = scaler.transform(X_train2)
  X_val_scaled = scaler.transform(X_val)
  model.fit(X_train2_scaled, y_train2)
  y_pred = model.predict(X_val_scaled)
  acc = accuracy_score(y_val, y_pred)
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"best k: {best_k}")
print(f"best accuracy: {best_accuracy}")

#Validação externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")

best k: 3
best accuracy: 1.0
test accuracy: 0.9444444444444444


In [189]:
### Não é a melhor forma. Não mesmo!
X_train_scaled = scaler.transform(X_train)
X_train2, X_val, y_train2, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2)

# Validação Interna
best_accuracy = 0
best_k = 0
for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(X_train2, y_train2)
  y_pred = model.predict(X_val)
  acc = accuracy_score(y_val, y_pred)
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"best k: {best_k}")
print(f"best accuracy: {best_accuracy}")
#Validação Externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")

best k: 3
best accuracy: 1.0
test accuracy: 0.9444444444444444


In [190]:
# A mesma coisa, mas com validação cruzada
from sklearn.model_selection import cross_validate, KFold

# Validação interna
spliter = KFold(n_splits=5, shuffle=True)
best_accuracy = 0
best_k = 0
for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  scores = cross_validate(model, X_train_scaled, y_train, cv=spliter)
  acc = scores['test_score'].mean()
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"best k: {best_k}")
print(f"best accuracy: {best_accuracy}")
# Validação externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")

best k: 9
best accuracy: 0.97192118226601
test accuracy: 0.9444444444444444


In [192]:
# Assim é a melhor forma! Com pipeline fica igual a primeira versão
from sklearn.pipeline import make_pipeline

# Validação interna
best_accuracy = 0
best_k = 0
for k in k_values:
  model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
  scores = cross_validate(model, X_train, y_train, cv=spliter)
  acc = scores['test_score'].mean()
  if acc > best_accuracy:
    best_accuracy = acc
    best_k = k

print(f"best k: {best_k}")
print(f"best accuracy: {best_accuracy}")
# Validação externa
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")

best k: 15
best accuracy: 0.9719211822660098
test accuracy: 0.9722222222222222


In [198]:
from sklearn.model_selection import GridSearchCV

# Validação interna
model = KNeighborsClassifier()
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}
grid_search = GridSearchCV(model, param_grid, cv=spliter)
grid_search.fit(X_train_scaled, y_train)
best_model = grid_search.best_estimator_
print(f"best k: {best_model.get_params()['n_neighbors']}")
print(f"best accuracy: {grid_search.best_score_}")
# Validação externa
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")
### Mas assim não é a melhor forma!

best k: 3
best accuracy: 0.9857142857142858
test accuracy: 0.9444444444444444


In [205]:
# Mas como usar gridsearch com pipeline

from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid = {'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}
grid_search = GridSearchCV(pipe, param_grid, cv=spliter)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print(f"best k: {best_model.get_params()['model__n_neighbors']}")
print(f"best accuracy: {grid_search.best_score_}")
# Validação externa
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
print(f"test accuracy: {accuracy_score(y_test, y_pred)}")
### Assim é a melhor forma!

best k: 9
best accuracy: 0.972167487684729
test accuracy: 0.9444444444444444


In [211]:
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"accuracy: {scores['test_score'].mean()}")

accuracy: 0.9665079365079364


In [213]:
grid_search = GridSearchCV(pipe, param_grid, cv=spliter, verbose=1)
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"accuracy: {scores['test_score'].mean()}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fitting 5 folds for each of 8 candidates, totalling 40 fits
accuracy: 0.9547619047619047


In [214]:
param_grid = {
    'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
    }
grid_search = GridSearchCV(pipe, param_grid, cv=spliter, verbose=1)
scores = cross_validate(grid_search, X, y, cv=spliter)
print(f"accuracy: {scores['test_score'].mean()}")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits
accuracy: 0.9606349206349206
